# Ollama Hallucination Experiments

Testing hallucination theory from "Why Do Language Models Hallucinate?" (Kalai et al., 2025) using real LLMs via Ollama.

This notebook demonstrates:
1. Setting up Ollama integration with Python
2. Testing the synthetic birthday dataset against real models
3. Measuring IIV classification vs generative error rates
4. Validating the paper's core claims

## Section 1: Setup and Installation

First, we'll ensure all dependencies are installed and the project is configured with `uv`.

In [ ]:
import sys
from pathlib import Path

# Add src to path for imports
src_path = Path.cwd() / "src"
if src_path.exists():
    sys.path.insert(0, str(src_path))

# Import the hallucination experiments package
try:
    from hallucination_experiments import (
        BirthdayDataset, 
        OllamaClient,
        ScoringScheme,
        evaluate_response
    )
    from hallucination_experiments.data import DatasetConfig
    from hallucination_experiments.experiments import HallucinationExperiment, ExperimentConfig
    print("✓ Imports successful")
except ImportError as e:
    print(f"Install with: pip install -e .")
    print(f"Error: {e}")
    
import json
print("\nSetup complete. Ready to run experiments.")

## Section 2: Initialize Ollama Client

Configure and test the Ollama connection.

In [ ]:
# Initialize Ollama client
# Make sure Ollama is running: ollama serve

MODEL = "llama2"  # Change to another model if desired
OLLAMA_HOST = "http://localhost:11434"

try:
    client = OllamaClient(model=MODEL, host=OLLAMA_HOST)
    print(f"Testing connection to {OLLAMA_HOST}...")
    
    if client.test_connection():
        print(f"✓ Connected to Ollama")
        print(f"✓ Model '{MODEL}' is available")
        print(f"\nModel info: {client.get_model_info()}")
    else:
        print(f"✗ Could not connect to Ollama")
        print(f"  Make sure it's running: ollama serve")
        print(f"  And model is pulled: ollama pull {MODEL}")
except Exception as e:
    print(f"Error: {e}")
    print(f"Instructions:")
    print(f"1. Install Ollama: https://ollama.ai/")
    print(f"2. Pull model: ollama pull {MODEL}")
    print(f"3. Run server: ollama serve")

## Section 3: Create Synthetic Birthday Dataset

Generate the synthetic dataset from the paper.

In [ ]:
# Create dataset configuration
config = DatasetConfig(
    n_people=200,      # Number of people
    n_dates=365,       # Possible birthdays
    n_docs=400,        # Documents in training corpus
    seed=42
)

# Generate dataset
dataset = BirthdayDataset(config)

# Print statistics
print("Birthday Dataset Created")
print("="*60)
dataset.print_stats()

print("\nKey insight from the paper:")
print(f"Singletons ({len(dataset.singletons)} people) were seen exactly ONCE")
print(f"in {config.n_docs} documents.")
print(f"The model CANNOT distinguish their true birthday from random dates!")
print(f"Therefore, singletons will be HALLUCINATED.")
print(f"\nSingleton rate: {dataset.singleton_rate:.1%}")
print(f"Theory: Generative error ≥ 2 × IIV classification error")

## Section 4: Test Model with Example Prompts

Quick tests to verify the model is responding to prompts.

In [ ]:
# Test basic generation
if client.test_connection():
    print("Testing model generation:")
    print("="*60)
    
    # Generate a birthday for a person from our dataset
    person_id = 0
    person_name = f"Person_{person_id}"
    true_date = dataset.get_person_birthday(person_id)
    mention_count = dataset.get_mention_count(person_id)
    
    print(f"\nPerson: {person_name}")
    print(f"True birthday: {true_date}")
    print(f"Mentions in training: {mention_count}")
    
    context = f"This person was mentioned {mention_count} times in training corpus"
    response = client.generate_birthday(person_name, context)
    
    print(f"\nModel response: {response}")
    
    # Test classification
    print("\n\nTesting model classification:")
    print("="*60)
    
    classification, conf = client.classify_birthday(person_name, str(true_date))
    print(f"\nIs the true date valid? {classification}")
    
    # Try with a wrong date
    wrong_date = (true_date + 100) % 365
    classification2, conf2 = client.classify_birthday(person_name, str(wrong_date))
    print(f"Is a random date valid? {classification2}")

## Section 5: Run Full Hallucination Experiments

Test the paper's main claims with the real model.

In [ ]:
# Configure experiment
exp_config = ExperimentConfig(
    model=MODEL,
    ollama_host=OLLAMA_HOST,
    dataset_config=config,
    n_test_samples=10,  # Adjust for more/fewer samples
    scoring_scheme=ScoringScheme.BINARY,
    penalty=1.0
)

# Create and run experiment
print("Starting Hallucination Experiments...")
print("="*60)

exp = HallucinationExperiment(exp_config)

# Run generation experiment (test if model hallucsinates on singletons)
print("\nExperiment 1: Generative Error on Singletons")
print("Hypothesis: Model should hallucinate on people mentioned only once in training")

try:
    exp.run_generation_experiment(n_samples=exp_config.n_test_samples)
except Exception as e:
    print(f"Note: This experiment requires Ollama to be running")
    print(f"Error: {e}")

In [ ]:
# Run IIV classification experiment
print("\n\nExperiment 2: Is-It-Valid (IIV) Binary Classification")
print("Hypothesis: Hard classification tasks lead to high generative error")

try:
    exp.run_iiv_classification_experiment(n_samples=exp_config.n_test_samples)
except Exception as e:
    print(f"Note: This experiment requires Ollama to be running")
    print(f"Error: {e}")

## Section 6: Summary and Results

Analyze the results from all experiments.

In [ ]:
print("\nEXPERIMENT SUMMARY")
print("="*60)

print("\nKey Findings from the Paper:")
print("-" * 60)
print("""
1. PRETRAINING CAUSES HALLUCINATION
   - Formula: Generative Error ≥ 2 × IIV Misclassification Rate
   - Singletons (seen once) are hallucinated at ~365x rate
   - The model cannot learn from one example
   
2. POST-TRAINING PRESERVES HALLUCINATION
   - Binary grading (1 if correct, 0 otherwise) incentivizes guessing
   - IDK gets 0 points, same as hallucination
   - Rational strategy: always guess rather than admit uncertainty
   
3. EVALUATION METRICS MATTER
   - Standard benchmarks reward guessing over honesty
   - Penalized grading (loss for wrong) would reduce hallucinations
""")

print("\nExperiment Results:")
print("-" * 60)
if exp.results:
    print(f"Total evaluations: {len(exp.results)}")
    metrics = exp.metrics.get_metrics()
    print(f"Accuracy: {metrics.get('accuracy', 0):.1%}")
    print(f"Hallucination rate: {metrics.get('hallucination_rate', 0):.1%}")
    print(f"Average score: {metrics.get('avg_score', 0):.3f}")
else:
    print("Run experiments above to see results")

print("\nNext Steps:")
print("-" * 60)
print("1. Try different models: mistral, neural-chat, openhermes")
print("2. Vary dataset size (--n-docs, --n-people)")
print("3. Test penalized scoring scheme")
print("4. Analyze patterns by mention frequency")

print("\nUseful Commands:")
print("-" * 60)
print("# Pull different models:")
print("  ollama pull mistral")
print("  ollama pull neural-chat")
print("  ollama pull openhermes")
print("\n# Run command-line experiments:")
print("  python scripts/run_experiments.py --model mistral --samples 30")
print("  python scripts/run_experiments.py --scoring penalized --penalty 2.0")